# Coverage-Constrained Price Range / 覆盖率约束价格区间研究

这个 notebook 的目标，是把 `coverage` 和 `range width` 的 trade-off 直接摊开来看，然后给出一个可解释的最优区间推荐。

核心问题是：

> 在给定 holding period 的情况下，怎样的价格区间既尽量窄，又不要太容易出区间？

这里的 `interval=1d/4h` 只是 data sampling frequency，不是最终要优化的 `price range / 价格区间`。

这份 notebook 会做四件事：

- 画单个 period 的 coverage frontier
- 做 `period x coverage target` 的交叉研究
- 给出每个 period 的最优区间推荐
- 给出当前参数下的整体最优区间推荐


In [ ]:
import pandas as pd
from IPython.display import Markdown, display

from app.research import (
    CoverageStudyRequest,
    CoverageSweepRequest,
    format_source,
    plot_coverage_frontier,
    plot_coverage_period_frontiers,
    plot_coverage_period_heatmap,
    rename_for_display,
    run_coverage_study,
    run_coverage_sweep,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)


## 参数

这里我先把默认研究参数设得更像正式调研：

- `holding_days_list` 包含 `7/30/60/180/365`
- `coverage_targets` 从 `50%` 开始往上扫，而不是只看 `70%+`
- `minimum_recommendation_coverage_pct = 80` 作为默认的推荐下限

这个 `80%` 不是数学真理，而是一个当前阶段比较实用的 floor：

- 低于这个值，区间虽然更窄，但 production 里太容易出区间
- 高于这个值，再去找最接近 `高 coverage + 低 width` 的点，会更接近你现在要的“最优区间”


In [ ]:
pair = "BTC/USDC"
interval = "1d"
days = 730

single_holding_days = 30
single_coverage_target = 90

holding_days_list = (7, 30, 60, 180, 365)
coverage_targets = (50.0, 60.0, 70.0, 80.0, 85.0, 90.0, 95.0, 97.5)
minimum_recommendation_coverage_pct = 80.0


In [ ]:
single_result = run_coverage_study(
    CoverageStudyRequest(
        pair=pair,
        interval=interval,
        days=days,
        holding_days=single_holding_days,
        coverage_target_pct=single_coverage_target,
        frontier_targets_pct=coverage_targets,
    )
)

sweep_result = run_coverage_sweep(
    CoverageSweepRequest(
        pair=pair,
        interval=interval,
        days=days,
        holding_days_list=holding_days_list,
        coverage_targets_pct=coverage_targets,
        minimum_recommendation_coverage_pct=minimum_recommendation_coverage_pct,
    )
)

display(
    Markdown(
        f"**Pair / 交易对:** {sweep_result.dataset.pair.symbol}  \n"
        f"**Sampling interval / 采样周期:** {sweep_result.dataset.interval}  \n"
        f"**Source / 数据源:** {format_source(sweep_result.dataset.source)}  \n"
        f"**Bars / 样本数:** {len(sweep_result.dataset.frame)}  \n"
        f"**Coverage sweep / coverage 扫描:** {coverage_targets[0]:.0f}% -> {coverage_targets[-1]:.1f}%  \n"
        f"**Recommendation floor / 推荐覆盖率下限:** {minimum_recommendation_coverage_pct:.0f}%"
    )
)

if sweep_result.dataset.notes:
    display(Markdown("**Notes / 说明**\n" + "\n".join(f"- {note}" for note in sweep_result.dataset.notes)))


## 单个 holding period 的 frontier

先固定一个 period，例如 `30 days`，看 coverage 提高时，区间宽度到底扩了多少。

这里最关键的是不要只看 `90%` 一个点，而是先看完整 frontier。这样你会知道：

- coverage 从 `50%` 提到 `80%` 时，宽度增加了多少
- 从 `80%` 再提到 `95%` 时，宽度是不是开始急剧变宽
- 哪个位置看起来像一个更合理的拐点


In [ ]:
single_frontier = sweep_result.frontier_grid[sweep_result.frontier_grid["holding_days"] == single_holding_days].copy()
rename_for_display(single_frontier[[
    "holding_days",
    "coverage_target_pct",
    "achieved_coverage_pct",
    "lower_bound_pct",
    "upper_bound_pct",
    "width_pct",
    "out_of_range_pct",
    "ideal_distance",
    "knee_score",
    "recommendation_eligible",
]])


In [ ]:
plot_coverage_frontier(
    single_result.frontier,
    title=f"{pair} {single_holding_days}d coverage vs range width frontier",
)


## Coverage x Width 交叉研究

这是这个 notebook 最重要的部分。现在我们不再只比较一个 `90%` 点，而是同时看：

- 不同 `holding period`
- 不同 `coverage target`
- 对应的 `width_pct`

如果某个 period 在 coverage 一上去时宽度迅速爆炸，那就说明这个 period 不适合拿来做窄区间。


In [ ]:
plot_coverage_period_heatmap(
    sweep_result.frontier_grid,
    metric="width_pct",
    title="Coverage x holding period width heatmap / 覆盖率与持有期的区间宽度热力图",
)


In [ ]:
plot_coverage_period_heatmap(
    sweep_result.frontier_grid,
    metric="out_of_range_pct",
    title="Coverage x holding period out-of-range heatmap / 覆盖率与持有期的出区间热力图",
    cmap="magma",
)


In [ ]:
plot_coverage_period_frontiers(
    sweep_result.frontier_grid,
    title=f"{pair} coverage frontiers across holding periods",
)


## 各个 period 的最优区间推荐

推荐逻辑是这样的：

1. 先把低于 `minimum_recommendation_coverage_pct` 的点排除掉
2. 在剩下的点里，选出最接近 `高 coverage + 低 width` 理想点的那个

所以这张表不是“绝对真理”，而是当前阶段一个比较清晰、可解释、可落地的推荐 heuristic。


In [ ]:
period_columns = [
    "holding_days",
    "coverage_target_pct",
    "achieved_coverage_pct",
    "lower_bound_pct",
    "upper_bound_pct",
    "width_pct",
    "center_offset_pct",
    "out_of_range_pct",
    "current_lower_price",
    "current_upper_price",
    "ideal_distance",
    "knee_score",
    "recommendation_reason",
]
rename_for_display(sweep_result.period_recommendations[period_columns])


In [ ]:
ax = sweep_result.period_recommendations.plot(
    x="holding_days",
    y="width_pct",
    marker="o",
    figsize=(8.5, 4.8),
    title="Recommended range width by holding period / 各 period 推荐区间宽度",
)
ax.set_xlabel("holding days / 持有期天数")
ax.set_ylabel("range width (%) / 区间宽度(%)")


## 当前整体最优区间推荐

在这一步里，我们把每个 period 的推荐点再拿出来比较一次，选出当前参数下的整体最优区间。

注意：

- 这一步默认没有加入 gas、rebalancing、人工盯盘成本
- 所以它会天然更偏向更短的 holding period
- 如果你后面把 operational cost 加进去，整体最优点可能会从 `7d` 往更长的 period 移动


In [ ]:
global_recommendation = pd.DataFrame([sweep_result.global_recommendation])
rename_for_display(global_recommendation[[
    "holding_days",
    "coverage_target_pct",
    "achieved_coverage_pct",
    "lower_bound_pct",
    "upper_bound_pct",
    "width_pct",
    "center_offset_pct",
    "out_of_range_pct",
    "current_lower_price",
    "current_upper_price",
    "ideal_distance",
    "knee_score",
    "recommendation_reason",
]])


## 怎么用这份结果

如果你现在还没有把 gas、重设频率、人工盯盘成本建进去，我建议你这样看：

- `7d` 代表更 aggressive、更勤快地重设区间
- `30d` 和 `60d` 更像中间方案
- `180d` 和 `365d` 更像长期静态区间的上限参考

实操上，通常不是直接盯着“全局最优区间”一条线用到底，而是：

- 先看各个 period 的推荐点
- 再结合你自己的 rebalancing 能力，决定你到底能接受多短的 holding period
- 最后再从那个 period 的推荐区间出发，继续做更细的 LP 回测
